<a href="https://colab.research.google.com/github/Dshah1003/CS4375-MLProject/blob/main/CS_4375_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install datasets transformers torch pandas -q

In [ ]:
from datasets import load_dataset
import pandas as pd

dataset = load_dataset("domenicrosati/TruthfulQA", split="train")
df = pd.DataFrame(dataset)

df = df[["Question", "Best Answer", "Correct Answers", "Incorrect Answers", "Category"]]
df.rename(columns={
    "Question": "question",
    "Best Answer": "best_answer",
    "Correct Answers": "correct_answers",
    "Incorrect Answers": "incorrect_answers",
    "Category": "category"
}, inplace=True)

print(f"✅ Loaded {len(df)} questions across {df['category'].nunique()} categories.")
df.head()

In [ ]:
from transformers import T5ForConditionalGeneration, T5Tokenizer
import torch

model_name = "google/flan-t5-base"
print(f"Loading model: {model_name}...")

tokenizer = T5Tokenizer.from_pretrained(model_name)
model = T5ForConditionalGeneration.from_pretrained(model_name)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

print(f"✅ Model loaded on {device}.")

In [ ]:
def generate_answer(question):
    prompt = f"Answer this question truthfully and to the best of your ability: {question}"
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=100,
        num_beams=4,
        early_stopping=True
    )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [ ]:
sample_df = df.copy()

answers = []
for i, question in enumerate(sample_df["question"].tolist()):
    print(f"[{i+1}/817] Generating...")
    answers.append(generate_answer(question))

sample_df["generated_answer"] = answers
sample_df[["question", "best_answer", "generated_answer"]]

In [ ]:
sample_df.to_csv("stage2_output.csv", index=False)
print("✅ Saved! Download it from the files -> content panel on the left.")